# Train SmolVLA from scratch

---

- Conda env : [lerobot](../README.md#setup-a-conda-environment)

----

- Ref: 
    - ...



### Device Setup

In [9]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Available device : {device}")

Available device : cuda


In [10]:
if device == "cuda":
    !nvidia-smi

Thu Jan  8 22:13:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 36%   44C    P8             31W /  250W |     588MiB /  11264MiB |     17%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## DataSet(svla-so101_pickplace) Visualization

In [11]:
!lerobot-dataset-viz \
    --repo-id lerobot/svla_so101_pickplace \
    --episode-index 0

[2026-01-09T06:13:22Z INFO  re_grpc_server] Listening for gRPC connections on 0.0.0.0:9876. Connect by running `rerun --connect rerun+http://127.0.0.1:9876/proxy`
[2026-01-09T06:13:22Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2026-01-09T06:13:23Z WARN  wgpu_hal::vulkan::instance] Unable to find extension: VK_EXT_physical_device_drm
[2026-01-09T06:13:23Z WARN  wgpu_hal::gles::egl] No config found!
[2026-01-09T06:13:23Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
  0%|                                                    | 0/10 [00:00<?, ?it/s][2026-01-09T06:13:23Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-01-09T06:13:23Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-01-09T06:13:23Z INFO  egui_wgpu] There were 3 available wgpu adapters: {backend: Vulkan, device_type: DiscreteGpu, name: "NVIDIA GeForce RTX 2080 Ti", 

## Fine-tuning SmolVAL with sval-so101-pickplace dataset

In [12]:
import os

output_dir = "./temp/outputs/svla_so101_pickplace_scratch"
print(output_dir)

./temp/outputs/svla_so101_pickplace_scratch


In [13]:
!lerobot-train \
    --policy.type=smolvla \
    --dataset.repo_id=lerobot/svla_so101_pickplace \
    --batch_size=8  \
    --steps=2000 \
    --save_freq=1000 \
    --eval_freq=10 \
    --policy.device=$device \
    --wandb.enable=false \
    --output_dir=./temp/outputs/svla_so101_pickplace_scratch \
    --policy.push_to_hub=false

INFO 2026-01-08 22:13:41 ot_train.py:163 {'batch_size': 8,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         't

## Preformace Test with the pre-trained model

In [ ]:
import os
import time
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from transformers import AutoProcessor

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load Policy
# Define output_dir if not already defined, or use a hardcoded path
output_dir = "./temp/outputs/smolvla_train" 
local_path = os.path.join(output_dir, "checkpoints/last/pretrained_model")

# Fallback for demonstration if local path doesn't exist
if not os.path.exists(local_path):
    print(f"Path '{local_path}' not found. Using 'lerobot/smolvla_base' for testing.")
    local_path = "lerobot/smolvla_base"

policy = SmolVLAPolicy.from_pretrained(local_path).to(device)
policy.eval()

# 3. Patch Tokenizer (Critical for SmolVLA inference)
if not hasattr(policy, "language_tokenizer") or policy.language_tokenizer is None:
    print("Patching missing language_tokenizer...")
    try:
        policy.language_tokenizer = AutoProcessor.from_pretrained(
            policy.config.vlm_model_name, trusted_remote_code=True
        ).tokenizer
    except Exception as e:
        print(f"Warning: Could not load tokenizer automatically: {e}")

# 4. Prepare Dummy Batch Dynamically
batch_size = 1
dummy_batch = {
    "task": ["stack the blocks"] * batch_size,
}

# --- A. Add State Dynamically ---
# We check the config to see what state dimension the model expects
state_feature = policy.config.input_features["observation.state"]
# Handle case where feature is an object (newer lerobot) or dict (older lerobot)
state_shape = state_feature.shape if hasattr(state_feature, "shape") else state_feature["shape"]
dummy_batch["observation.state"] = torch.rand(batch_size, state_shape[0], device=device)

# --- B. Add Images Dynamically ---
# We check which cameras the model was trained on (e.g., 'camera1', 'wrist')
if hasattr(policy.config, "image_features"):
    for cam_key, feature in policy.config.image_features.items():
        cam_shape = feature.shape if hasattr(feature, "shape") else feature["shape"]
        dummy_batch[cam_key] = torch.rand(batch_size, *cam_shape, device=device)
        print(f"Added dummy input for: {cam_key} with shape {cam_shape}")

# --- C. Pre-tokenize Language Inputs ---
# The select_action method expects tokenized inputs, not raw strings.
if "task" in dummy_batch and policy.language_tokenizer is not None:
    print("Tokenizing task instructions...")
    
    # Determine max token length from config (default to 64 if not found)
    max_len = 64
    if "observation.language.tokens" in policy.config.input_features:
        feat = policy.config.input_features["observation.language.tokens"]
        shape = feat.shape if hasattr(feat, "shape") else feat["shape"]
        max_len = shape[0]

    tokens = policy.language_tokenizer(
        dummy_batch["task"],
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    
    dummy_batch["observation.language.tokens"] = tokens["input_ids"].to(device)
    # CRITICAL FIX: Cast mask to bool to avoid RuntimeError in model attention
    dummy_batch["observation.language.attention_mask"] = tokens["attention_mask"].to(device).bool()

# 5. Warmup
# We use select_action() which handles normalization and internal state preparation
print("\nWarming up...")
for _ in range(3):
    with torch.no_grad():
        _ = policy.select_action(dummy_batch)

# 6. Benchmark
print("Benchmarking...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

start = time.time()
for _ in range(100):
    with torch.no_grad():
        # select_action returns the unnormalized action ready for the robot
        _ = policy.select_action(dummy_batch)

if torch.cuda.is_available():
    torch.cuda.synchronize()
end = time.time()

# 7. Results
avg_time = (end - start) / 100
print(f"Avg inference time: {avg_time:.6f} s ({1/avg_time:.2f} Hz)")
if torch.cuda.is_available():
    print(f"Max GPU memory used: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...
Loading weights from local directory
Patching missing language_tokenizer...
Added dummy input for: observation.images.camera1 with shape (3, 256, 256)
Added dummy input for: observation.images.camera2 with shape (3, 256, 256)
Added dummy input for: observation.images.camera3 with shape (3, 256, 256)
Added dummy input for: observation.images.empty_camera_0 with shape (3, 480, 640)
Tokenizing task instructions...

Warming up...
Benchmarking...
Avg inference time: 0.010771 s (92.84 Hz)
Max GPU memory used: 2196.65 MB


: 